In [1]:
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
import tensorflow as tf

# Load trained model (modify if needed)
trained_model = tf.keras.models.load_model("model_resnet50.h5")


In [2]:
from sklearn.decomposition import PCA
import numpy as np

# Load extracted features
feature_files = ["freshapples_features.npy", "freshbanana_features.npy", "freshoranges_features.npy",
                 "rottenapples_features.npy", "rottenbanana_features.npy", "rottenoranges_features.npy"]

all_features = []
labels = []

for idx, file in enumerate(feature_files):
    features = np.load(file)  # Load saved features
    all_features.append(features)
    labels.extend([idx] * len(features))  # Assign numeric labels to each class

# Stack all feature vectors
X = np.vstack(all_features)
y = np.array(labels)

# Apply PCA to reduce dimensions (e.g., to 100)
pca = PCA(n_components=100)
X_pca = pca.fit_transform(X)

# Save reduced features
np.save("pca_features.npy", X_pca)
np.save("labels.npy", y)

print("PCA applied. Shape after reduction:", X_pca.shape)


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 1417472 and the array at index 4 has size 0

In [3]:
for file in feature_files:
    features = np.load(file)
    print(f"{file} shape: {features.shape}")



freshapples_features.npy shape: (100, 1417472)
freshbanana_features.npy shape: (100, 1417472)
freshoranges_features.npy shape: (100, 1417472)
rottenapples_features.npy shape: (100, 1417472)
rottenbanana_features.npy shape: (0,)
rottenoranges_features.npy shape: (0,)


In [4]:
all_features = []
labels = []

for idx, file in enumerate(feature_files):
    features = np.load(file)
    
    if features.size == 0:  # Skip empty files
        print(f"Warning: {file} is empty. Skipping...")
        continue

    all_features.append(features)
    labels.extend([idx] * len(features))

# Proceed if we have valid features
if len(all_features) > 0:
    X = np.vstack(all_features)
    y = np.array(labels)

    # Apply PCA
    pca = PCA(n_components=100)
    X_pca = pca.fit_transform(X)

    # Save reduced features
    np.save("pca_features.npy", X_pca)
    np.save("labels.npy", y)

    print("PCA applied. Shape after reduction:", X_pca.shape)
else:
    print("Error: No valid features found. PCA not performed.")


PCA applied. Shape after reduction: (400, 100)


In [2]:
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
import tensorflow as tf

# Load trained model (modify if needed)
trained_model = tf.keras.models.load_model("model_efficient_3.h5")

In [10]:
from sklearn.decomposition import IncrementalPCA
import numpy as np

# List of feature files
feature_files = ["freshapples_features_2.npy", "freshbanana_features_2.npy", "freshoranges_features_2.npy",
                 "rottenapples_features_2.npy", "rottenbanana_features_2.npy", "rottenoranges_features_2.npy"]

batch_size = 10  # Adjust based on memory
n_components = 100
labels = []

# Initialize Incremental PCA
pca = IncrementalPCA(n_components=n_components)

# First pass: Fit PCA incrementally in small batches
for idx, file in enumerate(feature_files):
    features = np.load(file, mmap_mode='r')  # Load in memory-mapped mode (reduces RAM usage)
    
    num_samples = features.shape[0]
    labels.extend([idx] * num_samples)

    for i in range(0, num_samples, batch_size):
        batch = features[i:i + batch_size].astype(np.float32)  # Convert only the batch
        pca.partial_fit(batch)  # Fit incrementally

# Second pass: Transform in batches and save progressively
with open("pca_features.npy", "wb") as f:
    for file in feature_files:
        features = np.load(file, mmap_mode='r')
        
        for i in range(0, features.shape[0], batch_size):
            batch = features[i:i + batch_size].astype(np.float32)
            reduced_batch = pca.transform(batch)
            np.save(f, reduced_batch)  # Save batch-wise to avoid high RAM usage

# Save labels
np.save("labels.npy", np.array(labels))

print("PCA applied successfully. Features saved in 'pca_features.npy'")


MemoryError: Unable to allocate 54.1 MiB for an array with shape (10, 1417472) and data type float32

In [4]:
for file in feature_files:
    features = np.load(file)
    print(f"{file} shape: {features.shape}")



freshapples_features_2.npy shape: (100, 1417472)
freshbanana_features_2.npy shape: (100, 1417472)
freshoranges_features_2.npy shape: (100, 1417472)
rottenapples_features_2.npy shape: (100, 1417472)
rottenbanana_features_2.npy shape: (100, 1417472)
rottenoranges_features_2.npy shape: (100, 1417472)


In [6]:
all_features = []
labels = []

for idx, file in enumerate(feature_files):
    features = np.load(file)
    
    if features.size == 0:  # Skip empty files
        print(f"Warning: {file} is empty. Skipping...")
        continue

    all_features.append(features)
    labels.extend([idx] * len(features))

# Proceed if we have valid features
if len(all_features) > 0:
    X = np.vstack(all_features)
    y = np.array(labels)

    # Apply PCA
    pca = PCA(n_components=100)
    X_pca = pca.fit_transform(X)

    # Save reduced features
    np.save("pca_features.npy", X_pca)
    np.save("labels.npy", y)

    print("PCA applied. Shape after reduction:", X_pca.shape)
else:
    print("Error: No valid features found. PCA not performed.")



MemoryError: Unable to allocate 541. MiB for an array with shape (141747200,) and data type float32

In [1]:
from sklearn.decomposition import IncrementalPCA
import numpy as np

feature_file = "freshapples_features_2.npy"

batch_size = 10
n_components = min(10, batch_size)
label_value = 0  # Fresh apples label

pca = IncrementalPCA(n_components=n_components)
features = np.load(feature_file, mmap_mode='r')
num_samples = features.shape[0]
labels = [label_value] * num_samples

for i in range(0, num_samples, batch_size):
    batch = features[i:i + batch_size].astype(np.float32)
    if batch.shape[0] < n_components:
        break
    pca.partial_fit(batch)

with open("pca_freshapples_mobl.npy", "wb") as f:
    for i in range(0, features.shape[0], batch_size):
        batch = features[i:i + batch_size].astype(np.float32)
        reduced_batch = pca.transform(batch)
        np.save(f, reduced_batch)

np.save("labels_freshapples_mobl.npy", np.array(labels))
print("PCA applied for Fresh Apples")




PCA applied for Fresh Apples


In [3]:
feature_file = "freshbanana_features_2.npy"

batch_size = 10
n_components = min(10, batch_size)
label_value = 1  # Fresh bananas label

pca = IncrementalPCA(n_components=n_components)
features = np.load(feature_file, mmap_mode='r')
num_samples = features.shape[0]
labels = [label_value] * num_samples

for i in range(0, num_samples, batch_size):
    batch = features[i:i + batch_size].astype(np.float32)
    if batch.shape[0] < n_components:
        break
    pca.partial_fit(batch)

with open("pca_freshbanana_mobl.npy", "wb") as f:
    for i in range(0, features.shape[0], batch_size):
        batch = features[i:i + batch_size].astype(np.float32)
        reduced_batch = pca.transform(batch)
        np.save(f, reduced_batch)

np.save("labels_freshbanana_mobl.npy", np.array(labels))
print("PCA applied for Fresh Bananas")




PCA applied for Fresh Bananas


In [5]:
feature_file = "freshoranges_features_2.npy"

batch_size = 10
n_components = min(10, batch_size)
label_value = 2  # Fresh oranges label

pca = IncrementalPCA(n_components=n_components)
features = np.load(feature_file, mmap_mode='r')
num_samples = features.shape[0]
labels = [label_value] * num_samples

for i in range(0, num_samples, batch_size):
    batch = features[i:i + batch_size].astype(np.float32)
    if batch.shape[0] < n_components:
        break
    pca.partial_fit(batch)

with open("pca_freshoranges_mobl.npy", "wb") as f:
    for i in range(0, features.shape[0], batch_size):
        batch = features[i:i + batch_size].astype(np.float32)
        reduced_batch = pca.transform(batch)
        np.save(f, reduced_batch)

np.save("labels_freshoranges_mobl.npy", np.array(labels))
print("PCA applied for Fresh Oranges")


PCA applied for Fresh Oranges


In [6]:
feature_file = "rottenapples_features_2.npy"

batch_size = 10
n_components = min(10, batch_size)
label_value = 3  # Rotten apples label

pca = IncrementalPCA(n_components=n_components)
features = np.load(feature_file, mmap_mode='r')
num_samples = features.shape[0]
labels = [label_value] * num_samples

for i in range(0, num_samples, batch_size):
    batch = features[i:i + batch_size].astype(np.float32)
    if batch.shape[0] < n_components:
        break
    pca.partial_fit(batch)

with open("pca_rottenapples_mobl.npy", "wb") as f:
    for i in range(0, features.shape[0], batch_size):
        batch = features[i:i + batch_size].astype(np.float32)
        reduced_batch = pca.transform(batch)
        np.save(f, reduced_batch)

np.save("labels_rottenapples_mobl.npy", np.array(labels))
print("PCA applied for Rotten Apples")




PCA applied for Rotten Apples


In [7]:
feature_file = "rottenbanana_features_2.npy"

batch_size = 10
n_components = min(10, batch_size)
label_value = 4  # Rotten bananas label

pca = IncrementalPCA(n_components=n_components)
features = np.load(feature_file, mmap_mode='r')
num_samples = features.shape[0]
labels = [label_value] * num_samples

for i in range(0, num_samples, batch_size):
    batch = features[i:i + batch_size].astype(np.float32)
    if batch.shape[0] < n_components:
        break
    pca.partial_fit(batch)

with open("pca_rottenbanana_mobl.npy", "wb") as f:
    for i in range(0, features.shape[0], batch_size):
        batch = features[i:i + batch_size].astype(np.float32)
        reduced_batch = pca.transform(batch)
        np.save(f, reduced_batch)

np.save("labels_rottenbanana_mobl.npy", np.array(labels))
print("PCA applied for Rotten Bananas")



PCA applied for Rotten Bananas


In [9]:
feature_file = "rottenoranges_features_2.npy"

batch_size = 10
n_components = min(10, batch_size)
label_value = 5  # Rotten oranges label

pca = IncrementalPCA(n_components=n_components)
features = np.load(feature_file, mmap_mode='r')
num_samples = features.shape[0]
labels = [label_value] * num_samples

for i in range(0, num_samples, batch_size):
    batch = features[i:i + batch_size].astype(np.float32)
    if batch.shape[0] < n_components:
        break
    pca.partial_fit(batch)

with open("pca_rottenoranges_mobl.npy", "wb") as f:
    for i in range(0, features.shape[0], batch_size):
        batch = features[i:i + batch_size].astype(np.float32)
        reduced_batch = pca.transform(batch)
        np.save(f, reduced_batch)

np.save("labels_rottenoranges_mobl.npy", np.array(labels))
print("PCA applied for Rotten Oranges")



PCA applied for Rotten Oranges


In [12]:
import numpy as np
from sklearn.decomposition import PCA

# Function to apply PCA to individual feature sets
def apply_pca(features, n_components=None):
    if n_components is None or n_components > min(features.shape):
        n_components = min(features.shape)  # Ensure valid PCA component count
    pca = PCA(n_components=n_components)
    reduced_features = pca.fit_transform(features)
    return reduced_features, pca

# Load extracted features for each class
class_labels = ["freshapples", "freshbanana", "freshoranges", "rottenapples", "rottenbanana", "rottenoranges"]

for class_name in class_labels:
    print(f"Processing PCA for {class_name}...")
    
    # Load extracted features
    features = np.load(f"{class_name}_features.npy")
    
    # Split features into early, middle, and final layers (assuming equal division)
    num_features = features.shape[1] // 3
    early_features = features[:, :num_features]
    middle_features = features[:, num_features:num_features*2]
    final_features = features[:, num_features*2:]
    
    # --- Case 1: Apply PCA to each layer separately and fuse ---
    early_pca, early_model = apply_pca(early_features, n_components=50)
    middle_pca, middle_model = apply_pca(middle_features, n_components=50)
    final_pca, final_model = apply_pca(final_features, n_components=50)
    
    # Fuse reduced features
    fused_pca_features = np.concatenate([early_pca, middle_pca, final_pca], axis=1)
    np.save(f"{class_name}_fused_pca.npy", fused_pca_features)
    print(f"Fused PCA features saved: {class_name}_fused_pca.npy (Shape: {fused_pca_features.shape})")
    
    # --- Case 2: Apply PCA to combined features ---
    combined_pca, combined_model = apply_pca(features, n_components=150)  # Ensure valid components
    np.save(f"{class_name}_combined_pca.npy", combined_pca)
    print(f"Combined PCA features saved: {class_name}_combined_pca.npy (Shape: {combined_pca.shape})")

print("PCA processing complete for all classes!")


Processing PCA for freshapples...
Fused PCA features saved: freshapples_fused_pca.npy (Shape: (100, 150))
Combined PCA features saved: freshapples_combined_pca.npy (Shape: (100, 100))
Processing PCA for freshbanana...
Fused PCA features saved: freshbanana_fused_pca.npy (Shape: (100, 150))
Combined PCA features saved: freshbanana_combined_pca.npy (Shape: (100, 100))
Processing PCA for freshoranges...
Fused PCA features saved: freshoranges_fused_pca.npy (Shape: (100, 150))
Combined PCA features saved: freshoranges_combined_pca.npy (Shape: (100, 100))
Processing PCA for rottenapples...
Fused PCA features saved: rottenapples_fused_pca.npy (Shape: (100, 150))
Combined PCA features saved: rottenapples_combined_pca.npy (Shape: (100, 100))
Processing PCA for rottenbanana...
Fused PCA features saved: rottenbanana_fused_pca.npy (Shape: (100, 150))
Combined PCA features saved: rottenbanana_combined_pca.npy (Shape: (100, 100))
Processing PCA for rottenoranges...
Fused PCA features saved: rottenora

In [15]:
import numpy as np
import os
from sklearn.decomposition import PCA
from tqdm import tqdm

# Define class labels
class_labels = ["freshapples", "freshbanana", "freshoranges", "rottenapples", "rottenbanana", "rottenoranges"]

# Function to apply PCA
def apply_pca(features, n_components):
    max_components = min(features.shape[0], features.shape[1])  # Limit PCA to valid range
    n_components = min(n_components, max_components)  # Ensure it doesn't exceed limits
    pca = PCA(n_components=n_components)
    reduced_features = pca.fit_transform(features)
    return reduced_features, pca

# Process each class
for class_name in class_labels:
    print(f"Processing PCA for {class_name} (MobileNet)...")

    # Load extracted features from the corresponding class file
    features = np.load(f"{class_name}_features_2.npy")  

    # Split into early, middle, and final layers
    split_1 = features.shape[1] // 3
    split_2 = 2 * split_1
    early_features = features[:, :split_1]
    middle_features = features[:, split_1:split_2]
    final_features = features[:, split_2:]

    # Define valid PCA component limit
    max_pca_components = min(features.shape[0], split_1)

    # --- Case 1: Apply PCA to each layer separately and fuse ---
    early_pca, _ = apply_pca(early_features, max_pca_components)
    middle_pca, _ = apply_pca(middle_features, max_pca_components)
    final_pca, _ = apply_pca(final_features, max_pca_components)

    # Fuse the PCA-transformed features
    fused_pca_features = np.concatenate([early_pca, middle_pca, final_pca], axis=1)
    np.save(f"{class_name}_mobilenet_fused_pca.npy", fused_pca_features)
    print(f"Fused PCA features saved: {class_name}_mobilenet_fused_pca.npy (Shape: {fused_pca_features.shape})")

    # --- Case 2: Apply PCA to the combined layer ---
    combined_pca, _ = apply_pca(features, min(features.shape[0], features.shape[1]))
    np.save(f"{class_name}_mobilenet_combined_pca.npy", combined_pca)
    print(f"Combined PCA features saved: {class_name}_mobilenet_combined_pca.npy (Shape: {combined_pca.shape})")

print("PCA processing complete for all classes (MobileNet)!")



Processing PCA for freshapples (MobileNet)...
Fused PCA features saved: freshapples_mobilenet_fused_pca.npy (Shape: (100, 300))
Combined PCA features saved: freshapples_mobilenet_combined_pca.npy (Shape: (100, 100))
Processing PCA for freshbanana (MobileNet)...
Fused PCA features saved: freshbanana_mobilenet_fused_pca.npy (Shape: (100, 300))
Combined PCA features saved: freshbanana_mobilenet_combined_pca.npy (Shape: (100, 100))
Processing PCA for freshoranges (MobileNet)...
Fused PCA features saved: freshoranges_mobilenet_fused_pca.npy (Shape: (100, 300))
Combined PCA features saved: freshoranges_mobilenet_combined_pca.npy (Shape: (100, 100))
Processing PCA for rottenapples (MobileNet)...
Fused PCA features saved: rottenapples_mobilenet_fused_pca.npy (Shape: (100, 300))
Combined PCA features saved: rottenapples_mobilenet_combined_pca.npy (Shape: (100, 100))
Processing PCA for rottenbanana (MobileNet)...
Fused PCA features saved: rottenbanana_mobilenet_fused_pca.npy (Shape: (100, 300))


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.preprocessing import image
import os
import glob
from tqdm import tqdm  # Progress bar
from collections import defaultdict

# Load the trained EfficientNetB0 model
model = load_model("model_efficient_3.h5")

# Get the base EfficientNetB0 model
base_model = model.get_layer("efficientnetb0")  # Ensure this layer name matches your model

# Define feature extractors from different depths of EfficientNetB0
early_feature_extractor = Model(inputs=base_model.input, outputs=base_model.get_layer("block1a_activation").output)
middle_feature_extractor = Model(inputs=base_model.input, outputs=base_model.get_layer("block3a_expand_activation").output)
final_feature_extractor = Model(inputs=base_model.input, outputs=base_model.get_layer("top_activation").output)

# Define test image path
test_images_path = "./dataset/test"

# Define class labels
class_labels = ["freshapples", "freshbanana", "freshoranges", "rottenapples", "rottenbanana", "rottenoranges"]

# Store selected image paths for each class
class_images = defaultdict(list)

# Collect images for each class
for class_name in class_labels:
    class_path = os.path.join(test_images_path, class_name)
    if os.path.exists(class_path):
        image_paths = glob.glob(os.path.join(class_path, "*.*"))  # Load image files only
        class_images[class_name] = image_paths[:100]  # Take first 100 images

# Extract and save features for each class
for class_name, img_list in class_images.items():
    all_features = []
    
    print(f" Extracting features for {class_name} ({len(img_list)} images)...")
    
    for img_path in tqdm(img_list, desc=f"Processing {class_name}"):
        try:
            # Load and preprocess image
            img = image.load_img(img_path, target_size=(224, 224))
            img_array = image.img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
            img_array = img_array / 255.0  # Normalize

            # Extract features
            early_features = early_feature_extractor.predict(img_array).flatten()
            middle_features = middle_feature_extractor.predict(img_array).flatten()
            final_features = final_feature_extractor.predict(img_array).flatten()

            # Combine features
            combined_features = np.concatenate([early_features, middle_features, final_features])
            all_features.append(combined_features)
        
        except Exception as e:
            print(f"Error processing {img_path}: {e}")

    # Convert to numpy array
    all_features = np.array(all_features)

    # Save features to a separate file for each class
    feature_filename = f"{class_name}_features_3.npy"
    np.save(feature_filename, all_features)

    print(f" Features saved: {feature_filename} (Shape: {all_features.shape})")

print(" Feature extraction complete for all classes!")


In [16]:
import numpy as np
from sklearn.decomposition import PCA
import os
from tqdm import tqdm

# Define class labels
class_labels = ["freshapples", "freshbanana", "freshoranges", "rottenapples", "rottenbanana", "rottenoranges"]

# Define number of components for PCA
pca_components = 100  # Adjust based on dimensionality and variance explained

# Perform PCA and save results
for class_name in class_labels:
    print(f"Processing PCA for {class_name}...")
    
    # Load features
    feature_filename = f"{class_name}_features_3.npy"
    if not os.path.exists(feature_filename):
        print(f"Skipping {class_name}, file not found: {feature_filename}")
        continue
    
    all_features = np.load(feature_filename)
    
    # Split features into early, middle, and final layers
    split_idx1 = all_features.shape[1] // 3
    split_idx2 = 2 * split_idx1
    
    early_features = all_features[:, :split_idx1]
    middle_features = all_features[:, split_idx1:split_idx2]
    final_features = all_features[:, split_idx2:]
    
    # Case 1: Apply PCA separately to each layer, then fuse them
    pca_early = PCA(n_components=pca_components)
    pca_middle = PCA(n_components=pca_components)
    pca_final = PCA(n_components=pca_components)
    
    early_pca = pca_early.fit_transform(early_features)
    middle_pca = pca_middle.fit_transform(middle_features)
    final_pca = pca_final.fit_transform(final_features)
    
    fused_pca = np.concatenate([early_pca, middle_pca, final_pca], axis=1)
    
    # Save fused PCA output
    fused_pca_filename = f"efficient_{class_name}_fused_pca.npy"
    np.save(fused_pca_filename, fused_pca)
    print(f"Saved fused PCA features: {fused_pca_filename} (Shape: {fused_pca.shape})")
    
    # Case 2: Apply PCA to the combined feature layer
    pca_combined = PCA(n_components=pca_components)
    combined_pca = pca_combined.fit_transform(all_features)
    
    # Save combined PCA output
    combined_pca_filename = f"efficient_{class_name}_combined_pca.npy"
    np.save(combined_pca_filename, combined_pca)
    print(f"Saved combined PCA features: {combined_pca_filename} (Shape: {combined_pca.shape})")

print("PCA processing complete for all classes!")

Processing PCA for freshapples...
Saved fused PCA features: efficient_freshapples_fused_pca.npy (Shape: (100, 300))
Saved combined PCA features: efficient_freshapples_combined_pca.npy (Shape: (100, 100))
Processing PCA for freshbanana...
Saved fused PCA features: efficient_freshbanana_fused_pca.npy (Shape: (100, 300))
Saved combined PCA features: efficient_freshbanana_combined_pca.npy (Shape: (100, 100))
Processing PCA for freshoranges...
Saved fused PCA features: efficient_freshoranges_fused_pca.npy (Shape: (100, 300))
Saved combined PCA features: efficient_freshoranges_combined_pca.npy (Shape: (100, 100))
Processing PCA for rottenapples...
Saved fused PCA features: efficient_rottenapples_fused_pca.npy (Shape: (100, 300))
Saved combined PCA features: efficient_rottenapples_combined_pca.npy (Shape: (100, 100))
Processing PCA for rottenbanana...
Saved fused PCA features: efficient_rottenbanana_fused_pca.npy (Shape: (100, 300))
Saved combined PCA features: efficient_rottenbanana_combined